# Adjudication → Pricing → Monitoring → Line Increase 

**A single study path through the credit analytics lifecycle**

This notebook connects the decisions made at origination to the questions asked after a
loan is booked. We reuse the **same score spine**, but the decision changes at each point:

- **Adjudication:** should we say yes, refer, or decline?
- **Pricing:** if we say yes, is the rate high enough to earn the hurdle?
- **Monitoring:** which booked accounts are getting worse, and where can we act?
- **Line increases:** who has earned more credit inside risk appetite?

The goal is not to memorize every output. It is to understand which analytical object is
being consumed by the next decision, and which metric tells you whether that decision is
useful.

## The flow of decisions

Read the diagram left to right. The score is a common input, but it is not the final
decision. Policy, economics, time, and operational capacity add the context needed to
turn analytics into action.

```mermaid
flowchart LR
    A[Applicant data<br/>score / PD / bands] --> B[Adjudication<br/>Approve · Refer · Decline]
    B -->|Approve| C[Book the loan]
    B -->|Refer| C
    C --> D[Pricing<br/>quoted rate vs ROE hurdle]
    D --> E[Portfolio over time<br/>behavior + utilization + delinquency]
    E --> F[Early warning<br/>who is getting worse?]
    E --> G[Line increase<br/>who can safely grow?]
    F --> H[Work the highest-risk<br/>short list]
    G --> I[Offer only when<br/>economics + risk pass]
```

**Study prompt:** at every arrow, ask “what did the previous stage produce, and what does
the next stage need?” That question is the thread connecting the four sections below.

## Before you run it

Run `python verify.py` first and use the `.venv` Python 3 kernel. This combined notebook
expects the completed stage-3 and stage-4 artifacts to be present. The original notebooks
remain available if you want to study one stage in isolation.

In [ ]:
# --- bootstrap: find the repo root, make the repo importable ---------------
# The modules use paths relative to the repo root (e.g. Path("score/models")),
# so we chdir there. This works whether you launched Jupyter from the repo
# root or from notebooks/.
import os, sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "verify.py").exists()), None)
assert root is not None, "Could not find the repo root (no verify.py above cwd)."
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})

print(f"repo root: {root}")
print(f"stage.txt: {(root / 'stage.txt').read_text().strip()}")

In [ ]:
# --- combined notebook stage guard ---------------------------------------
NEEDS = [
    "adjudication/models/adjudication_model.pkl",
    "adjudication/models/policy_config.json",
    "ews/models/metadata.json",
    "line_increase/models/metadata.json",
]
missing = [p for p in NEEDS if not Path(p).exists()]
if missing:
    raise SystemExit(
        "This notebook needs stage-3 and stage-4 artifacts. Missing:\n  "
        + "\n  ".join(missing)
        + "\n\nYou are at stage " + Path("stage.txt").read_text().strip()
        + ". Fix with either:\n"
        "  git checkout stage-4      # jump to the finished stage, or\n"
        "  make <the stage's build step>  # build it yourself (see the lab sheet)"
    )
print("stage-3 and stage-4 artifacts present.")

## 1. Adjudication — a score is not a decision

The model produces a probability of default (PD). The **policy layer** turns that
probability into an operational decision with an explanation. A bank usually separates
model risk ranking from policy rules so that risk appetite, affordability, legal controls,
and concentration limits remain visible and governable.

### The decision sequence

```mermaid
flowchart TD
    A[Application data + PD] --> B{Hard knockout?}
    B -->|Yes| D[Decline + reason trail]
    B -->|No| C{PD zone}
    C -->|Low PD| E[Approve zone]
    C -->|Middle PD| F[Refer]
    C -->|High PD| D
    E --> G{Refer override?}
    G -->|Yes| F
    G -->|No| H[Approve]
```

1. **Hard knockouts** are non-negotiable policy failures. They can decline an applicant
   before the model votes: for example, insufficient debt-service capacity, public records,
   `prior_delinquencies`, severe prior delinquency, or excessive leverage.
2. **PD zones** map the model output to an initial action: low PD → Approve, middle PD →
   Refer, high PD → Decline. The thresholds are calibrated to the product and risk appetite.
3. **Refer overrides** only downgrade an approval to Refer when another concern appears;
   they do not quietly upgrade a Decline. This preserves a human review path without
   allowing the policy layer to hide hard failures.

In this workshop, the policy configuration includes `t_low`, `t_high`, `dscr_floor`,
`dscr_refer_hi`, `public_records_cap`, `prior_delinq_cap`, `leverage_cap`,
`req_to_rev_cap` (the `requested_amount / annual_revenue` limit), and `score_floor`.
These are teaching values, not universal bank standards; real thresholds are calibrated by
product, portfolio performance, regulation, and risk appetite.

### Typical bank parameters for credit approval

Banks usually combine model outputs with several parameter families. The exact cutoff is
product-specific, but the business question is consistent: **can this borrower repay, what
could make the exposure unsafe, and what controls must stop or escalate it?**

| Parameter family | Typical examples | How it influences the decision |
| --- | --- | --- |
| Affordability and cash flow | DSCR, interest coverage, free cash flow, debt-service burden | Approve only when operating cash flow can support the proposed debt; borderline cases are referred. |
| Credit history and bureau | bureau score, prior delinquencies, utilization, public records, bankruptcies | Measures repayment behavior and may create hard knockouts or PD adjustments. |
| Leverage and capacity | debt-to-revenue, debt-to-equity, leverage, requested amount / annual revenue | Limits how much debt the borrower can carry relative to its business scale. |
| Collateral and recovery | collateral value, loan-to-value, guarantees, lien position, collateral type | Reduces expected loss or supports a lower exposure when the borrower defaults. |
| Exposure and concentration | requested amount, total relationship exposure, industry/region limits, single-name caps | Prevents one borrower or segment from dominating the portfolio. |
| KYC, AML, and fraud | identity, beneficial ownership, sanctions, unusual application patterns, fraud alerts | Usually a stop or escalation control before normal credit scoring. |
| Business and policy context | time in business, revenue, industry, purpose, product/term, exceptions | Applies the bank's target-market and risk-appetite rules. |

The first output to inspect is therefore not AUC. It is the decision mix and the reason
trail. Those tell you how policy changes who enters the book and whether the decision can be
explained to a customer, credit officer, or validator.

In [ ]:
import json, joblib
from shared.config import RAW
from adjudication.src.feature_engineering import (
    ADJ_FEATURE_COLUMNS, compute_adjudication_features)
from adjudication.src.policy import PolicyConfig, decide
from score.src.predict import predict_score_pd

businesses = pd.read_parquet(RAW / "businesses.parquet")
model  = joblib.load("adjudication/models/adjudication_model.pkl")
config = PolicyConfig.from_dict(json.loads(Path("adjudication/models/policy_config.json").read_text()))

X_all  = compute_adjudication_features(businesses)
pd_hat = model.predict_proba(X_all[ADJ_FEATURE_COLUMNS])[:, 1]
dec    = decide(X_all, pd_hat, config)
dec["score_band"] = predict_score_pd(businesses)["score_band"].to_numpy()

mix = dec["decision"].value_counts().reindex(["Approve", "Refer", "Decline"]).fillna(0)
COL = {"Approve": "#2f7d4f", "Refer": "#c9a227", "Decline": "#b3372e"}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(mix.index, mix.values, color=[COL[k] for k in mix.index])
for i, v in enumerate(mix.values):
    axes[0].text(i, v + 60, f"{int(v):,}\n{v/len(dec):.1%}", ha="center", weight="bold")
axes[0].set_title(f"Decisions across {len(dec):,} applicants"); axes[0].set_ylabel("applicants")

ct = (pd.crosstab(dec["score_band"], dec["decision"], normalize="index")
        .reindex(["D", "C", "B", "A", "AAA"]).dropna(how="all"))
ct = ct[[c for c in ["Approve", "Refer", "Decline"] if c in ct.columns]]
ct.plot(kind="barh", stacked=True, ax=axes[1], color=[COL[c] for c in ct.columns], width=.75)
axes[1].set_title("Decision mix by score band"); axes[1].set_xlabel("share")
axes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# Make the lifecycle join explicit: adjudication preserves the source row order,
# while downstream portfolio modules expose business_id as a column/index.
dec = dec.copy()
dec["business_id"] = businesses["business_id"].to_numpy()

### What to notice

- The same score can lead to different outcomes when a policy rule fires.
- A decision mix is a portfolio choice, not just a model statistic.
- A reason trail makes the decision reviewable: a student, customer, or validator should
  be able to ask “why?” and get a named answer.

Next, inspect the reasons rather than treating `Refer` and `Decline` as unexplained labels.

In [ ]:
from collections import Counter
reasons = Counter(r for rs in dec["decision_reasons"] for r in (rs if isinstance(rs, list) else [rs]) if r)
top = pd.Series(dict(reasons.most_common(10))).sort_values()

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.barh(top.index, top.values, color="#5b8ac6")
ax.set_xlabel("applicants"); ax.set_title("Most common decision reasons")
plt.tight_layout(); plt.show()
top.sort_values(ascending=False).to_frame("applicants")

## 2. Pricing — arithmetic turns risk into economics

Pricing consumes the booked population and its PD. It does not need another machine
learning model. The engine asks whether the quoted rate covers the full economic stack:

```text
interest income
− cost of funds
− expected loss (PD × LGD)
− operating cost
= pre-tax profit → net income → ROE
```

### ROE and RAROC: two views of the same loan economics

- **ROE (return on equity)** = after-tax net income ÷ allocated equity. It answers:
  “Does this loan earn the bank's required return on the capital tied up in it?” The
  workshop hurdle is **15% ROE**.
- **RAROC (risk-adjusted return on capital)** = pre-tax profit ÷ allocated equity. It
  includes expected loss in the profit calculation, but before tax, so it is useful for
  comparing risk-adjusted economics across products or customer segments.

The simplified capital allocation is `allocated equity = capital ratio × EAD`. These
metrics support business decisions, not just model reporting:

- **Below the hurdle:** the quoted rate does not compensate the bank for funding, expected
  loss, operating cost, taxes, and allocated capital. Reprice, change the structure, or do
  not book the exposure.
- **Above the hurdle but below recommended:** the loan earns the minimum capital return,
  but does not include the extra pricing cushion.
- **Above recommended:** the loan clears the hurdle and the reference margin cushion.

The **break-even rate** makes ROE zero. The **hurdle-clearing rate** makes ROE reach the
target. The recommended rate adds margin. This is why a simple, auditable calculation can
produce a larger business finding than a more sophisticated model.

In [ ]:
from pricing.src.portfolio import price_population, portfolio_summary
from pricing.src.engine import (MarketAssumptions, break_even_rate,
                                hurdle_clearing_rate, recommended_rate, profit_waterfall)
from shared.config import MARKET

priced = price_population()
summary = portfolio_summary(priced)

print(f"booked loans priced      : {summary['n']:,}")
print(f"clear the ROE hurdle     : {summary['n_clears']:,}  ({summary['share_clears']:.1%})")
print(f"median ROE at quoted rate: {summary['median_roe']:.2%}")
print(f"MISPRICED EXPOSURE       : ${summary['mispriced_ead']:,.0f}")

In [ ]:
m = MarketAssumptions.from_market(MARKET)
pds = np.linspace(0.005, 0.25, 120)
ead = float(priced["ead"].median())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(pds * 100, [break_even_rate(p, ead, m) * 100 for p in pds], label="break-even")
axes[0].plot(pds * 100, [hurdle_clearing_rate(p, ead, m) * 100 for p in pds], label="clears hurdle")
axes[0].plot(pds * 100, [recommended_rate(p, ead, m) * 100 for p in pds], label="recommended", lw=2.5)
axes[0].set_xlabel("PD (%)"); axes[0].set_ylabel("rate (%)")
axes[0].set_title("Price is a function of risk"); axes[0].legend()

axes[1].hist(priced["roe_at_quoted"] * 100, bins=50, color="#9fb4d4", edgecolor="white")
axes[1].axvline(m.roe_hurdle * 100, color="#b3372e", ls="--", lw=2,
                label=f"hurdle {m.roe_hurdle:.0%}")
axes[1].set_xlabel("ROE at the quoted rate (%)"); axes[1].set_title("Who actually earns their cost of capital?")
axes[1].legend()
plt.tight_layout(); plt.show()

### What to notice

- A loan can have a low PD and still be underpriced if the quoted rate cannot cover the
  capital and operating floor.
- Compare `roe_at_quoted` with the hurdle, not only with the model's PD.
- The question is not “is the model accurate?” but “did the decision produce acceptable
  economics for this risk?”

In [ ]:
mis = priced[priced["mispriced"]]
by_band = (priced.groupby("score_band", observed=True)
           .apply(lambda g: pd.Series({
               "loans": len(g),
               "mispriced_loans": int(g["mispriced"].sum()),
               "mispriced_ead": float(g.loc[g["mispriced"], "ead"].sum())}),
                  include_groups=False)
           .reindex(["D", "C", "B", "A", "AAA"]).dropna(how="all"))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_band.index, by_band["mispriced_ead"] / 1e6, color="#b3372e")
ax.set_ylabel("mispriced exposure ($m)"); ax.set_xlabel("score band")
ax.set_title(f"Mispriced exposure by band — ${summary['mispriced_ead']/1e9:.2f}B total")
plt.tight_layout(); plt.show()

by_band.style.format({"loans": "{:,.0f}", "mispriced_loans": "{:,.0f}", "mispriced_ead": "${:,.0f}"})

In [ ]:
# Compare three real booked loans with different pricing outcomes.
# The categories are selected from the same priced population, so the comparison is not a toy example.
priced_cases = priced.copy()
priced_cases["hurdle_rate"] = [
    hurdle_clearing_rate(float(pd_), float(ead_), m)
    for pd_, ead_ in zip(priced_cases["pd"], priced_cases["ead"])
]
priced_cases["recommended_rate"] = [
    recommended_rate(float(pd_), float(ead_), m)
    for pd_, ead_ in zip(priced_cases["pd"], priced_cases["ead"])
]

case_specs = [
    (
        "Below hurdle",
        priced_cases.loc[priced_cases["quoted_rate"] < priced_cases["hurdle_rate"]]
        .sort_values("ead", ascending=False).iloc[0],
    ),
    (
        "Above hurdle / below recommended",
        priced_cases.loc[
            (priced_cases["quoted_rate"] >= priced_cases["hurdle_rate"])
            & (priced_cases["quoted_rate"] < priced_cases["recommended_rate"])
        ].sort_values("ead", ascending=False).iloc[0],
    ),
    (
        "Above recommended",
        priced_cases.loc[priced_cases["quoted_rate"] >= priced_cases["recommended_rate"]]
        .sort_values("ead", ascending=False).iloc[0],
    ),
]

def summarize_case(row):
    waterfall = profit_waterfall(float(row["pd"]), float(row["ead"]), float(row["quoted_rate"]), m)
    return pd.Series({
        "business_id": row["business_id"],
        "PD": row["pd"],
        "EAD": row["ead"],
        "quoted rate": row["quoted_rate"],
        "hurdle rate": row["hurdle_rate"],
        "recommended rate": row["recommended_rate"],
        "ROE at quoted": waterfall["roe"],
        "RAROC at quoted": waterfall["raroc"],
        "business decision": {
            "Below hurdle": "Reprice, restructure, or do not book",
            "Above hurdle / below recommended": "Acceptable minimum return; review margin",
            "Above recommended": "Clears hurdle plus pricing cushion",
        }[row["case_label"]],
    })

# Add labels only for the display helper; the source rows remain unchanged.
cases = []
for label, row in case_specs:
    row = row.copy()
    row["case_label"] = label
    cases.append((label, row))

comparison = pd.DataFrame({label: summarize_case(row) for label, row in cases})
comparison.style.format({
    "PD": "{:.2%}",
    "EAD": "${:,.0f}",
    "quoted rate": "{:.2%}",
    "hurdle rate": "{:.2%}",
    "recommended rate": "{:.2%}",
    "ROE at quoted": "{:.2%}",
    "RAROC at quoted": "{:.2%}",
})

## The handoff: from booked loan to living portfolio

At this point, adjudication and pricing have created a booked portfolio. From here onward,
the unit of analysis changes: we observe each account repeatedly over time. Application
features explain how the relationship started; behavioral features explain what is
happening now.

That change in time horizon is the reason monitoring uses different metrics and different
operational questions.

## 3. Monitoring — find tomorrow's trouble with a workable list

An early-warning system does not need to perfectly predict every future deterioration. A
relationship manager has limited capacity, so the operational product is a short list:

> If we can work only 10% of the book, how much of tomorrow's trouble lands in that 10%?

That is **top-decile capture**. AUC is still useful as a reported ranking statistic, but it
is not the gate here because deterioration is noisy and the data has a ceiling. Always
quote the held-out capture number when describing real-world performance.

### How deterioration probability is modeled

The Early Warning System predicts the target `deterioration_next_6_12mo`: whether an
already-booked account deteriorates in the next 6–12 months. This is different from the
origination default model because the account now has behavioral history.

The model uses three feature groups:

1. **Behavioral trends from the 24-month panel:** recent utilization, utilization drift,
   balance and deposit trends, days past due, and overdraft activity.
2. **Current account and firm information:** utilization, balance, credit limit, tenure,
   DSCR, leverage, current ratio, prior delinquencies, industry, and entity type.
3. **Risk-spine context:** the Session 2 `business_score` and `pd_score` are reused, but
   the target is new and downstream.

Training follows an 80/20 stratified `train_test_split`. The training script fits a
LightGBM classifier with 500 trees, learning rate `0.03`, 31 leaves, and row/column
subsampling. The saved model's `predict_proba(... )[:, 1]` is the estimated probability of
deterioration. The full-population probability distribution then supplies the High and
Medium risk-tier cutoffs; named behavioral triggers provide reasons for the alert.

The next cell shows the training recipe, the feature groups, and one real saved-model
prediction. It demonstrates probability generation without retraining the model.

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from ews.src.feature_engineering import EWS_FEATURE_COLUMNS, compute_ews_features
from ews.src.watchlist import risk_tier

demo_portfolio = pd.read_parquet(RAW / "portfolio.parquet")
demo_y = demo_portfolio["deterioration_next_6_12mo"].to_numpy()
demo_train_idx, demo_test_idx = train_test_split(
    np.arange(len(demo_portfolio)), test_size=0.20, random_state=42, stratify=demo_y
)

feature_groups = {
    "behavioral trends": [
        "util_recent", "util_drift", "balance_recent", "deposit_recent",
        "deposit_decline_pct", "dpd_months", "dpd_max", "dpd_recent",
        "overdraft_total", "overdraft_recent",
    ],
    "account and firm": [
        "utilization_onbook", "current_balance", "credit_limit", "tenure_months",
        "dscr", "leverage", "current_ratio", "prior_delinquencies",
        "industry", "entity_type",
    ],
    "risk-spine context": ["business_score", "pd_score"],
}

training_recipe = {
    "target": "deterioration_next_6_12mo",
    "train_rows": len(demo_train_idx),
    "test_rows": len(demo_test_idx),
    "base_rate": demo_y.mean(),
    "model": "LightGBM classifier",
    "trees": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "subsampling": "row=0.80, columns=0.80",
}
print(json.dumps(training_recipe, indent=2, default=float))

demo_model = joblib.load("ews/models/ews_model.pkl")
demo_features = compute_ews_features(demo_portfolio)
demo_row = demo_features.iloc[[0]]
demo_prob = float(demo_model.predict_proba(demo_row[EWS_FEATURE_COLUMNS])[:, 1][0])
demo_meta = json.loads(Path("ews/models/metadata.json").read_text())
demo_tier = risk_tier(demo_prob, demo_meta["tiers"])

print(f"\nexample business: {demo_row['business_id'].iloc[0]}")
print(f"predicted deterioration probability: {demo_prob:.1%}")
print(f"risk tier from saved cutoffs: {demo_tier}")

pd.DataFrame([
    {"feature group": group, "features": ", ".join(features)}
    for group, features in feature_groups.items()
])

In [ ]:
import json
from ews.src import watchlist as ews_watchlist

ews_meta = json.loads(Path("ews/models/metadata.json").read_text())
print("metrics:", json.dumps(ews_meta["metrics"], indent=2))
print("gate   :", json.dumps(ews_meta["gate"], indent=2))

ews = ews_watchlist.score_population()
tiers = ews["risk_tier"].value_counts().reindex(["High", "Medium", "Low"]).fillna(0)
tiers.to_frame("accounts")

### What to notice

- The model's probability is a ranking signal; the watchlist tier is an operating decision.
- A named trigger explains why an account was surfaced, which makes the list actionable.
- The held-out capture is the honest estimate of what the workflow can find on new data.

Inspect both the tier counts and the trigger reasons before looking at the curve.

In [ ]:
COLT = {"High": "#b3372e", "Medium": "#c9a227", "Low": "#2f7d4f"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(tiers.index, tiers.values, color=[COLT[t] for t in tiers.index])
for i, v in enumerate(tiers.values):
    axes[0].text(i, v + 60, f"{int(v):,}", ha="center", weight="bold")
axes[0].set_title("Watchlist tiers"); axes[0].set_ylabel("accounts")

axes[1].hist(ews["prob"], bins=50, color="#9fb4d4", edgecolor="white")
axes[1].set_xlabel("p(deterioration)"); axes[1].set_title("Deterioration probability")
plt.tight_layout(); plt.show()

In [ ]:
from collections import Counter
trig = Counter(t for ts in ews["triggers"] for t in ts)
s = pd.Series(dict(trig)).sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(s.index, s.values, color="#5b8ac6")
ax.set_xlabel("accounts flagged"); ax.set_title("How often each named trigger fires")
plt.tight_layout(); plt.show()

top10 = (ews.sort_values("prob", ascending=False).head(10)
         [["business_id", "risk_tier", "prob", "triggers"]])
top10["triggers"] = top10["triggers"].apply(lambda t: ", ".join(t))
top10.reset_index(drop=True).style.format({"prob": "{:.1%}"})

In [ ]:
cap  = ews_meta["metrics"]["top_decile_capture"]
lift = ews_meta["metrics"]["top_decile_lift"]
gate = ews_meta["gate"]["capture_min"]

print(f"top-decile capture (held out): {cap:.1%}")
print(f"                      = lift : {lift:.2f}x random")
print(f"gate                         : >= {gate:.0%}")
print(f"result                       : {'PASS' if cap >= gate else 'FAIL'}")
print(f"\nWork 10% of the book, catch {cap:.1%} of the deteriorations.")

### Why in-sample capture can be higher

The blue whole-book curve can show roughly **54% capture at 10% reviewed**, while the
held-out red point is about **21%**. This is a generalization gap, not a contradiction.

- The blue curve is calculated after scoring the whole portfolio with the saved model. It
  includes accounts that were used during training, so it is an optimistic, partly
  **in-sample** diagnostic. The model has already seen patterns from those accounts and can
  rank training-specific noise unusually well.
- The red point uses predictions for an unseen held-out test split. Those accounts and
  their outcomes were not available when the model learned its ranking. The approximately
  21% capture is therefore the more honest estimate of what a future watchlist may catch.
- A model can be leakage-safe and still overfit. Removing the target from the features
  prevents direct cheating; held-out evaluation checks whether the learned relationship
  generalizes beyond the training rows.

The operational decision should use the held-out number, not the attractive whole-book
curve. That is why the training gate is set on held-out capture (at least 20%, or 2× random
lift), and why the chart labels the red point as the number to quote.

In [ ]:
from shared.config import RAW
port = pd.read_parquet(RAW / "portfolio.parquet").set_index("business_id")
truth = port["deterioration_next_6_12mo"].reindex(ews["business_id"]).to_numpy()

order = np.argsort(-ews["prob"].to_numpy())
y = truth[order]
xs = np.linspace(.05, 1, 20)
curve = [y[:max(1, int(len(y) * q))].sum() / y.sum() for q in xs]
in_sample_10 = y[:int(len(y) * .1)].sum() / y.sum()

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(xs * 100, np.array(curve) * 100, marker="o", color="#245bb2", label="in-sample (optimistic)")
ax.plot([0, 100], [0, 100], ls="--", color="#999", label="random")
ax.scatter([10], [cap * 100], s=90, color="#b3372e", zorder=5,
           label=f"held-out at 10% = {cap:.1%}")
ax.axvline(10, color="#b3372e", ls=":", lw=1.2)
ax.set_xlabel("% of book reviewed (worst first)"); ax.set_ylabel("% of deteriorations caught")
ax.set_title("Capture curve — quote the red dot, not the blue line")
ax.legend(); plt.tight_layout(); plt.show()

print(f"in-sample at 10%: {in_sample_10:.1%}   vs   held-out: {cap:.1%}")
print("The difference is why we gate on the held-out number.")

## 4. Line increases — growth inside risk appetite

Monitoring can identify customers who may deserve more credit, but the offer engine is a
**conjunctive gate**: every condition below must pass. There is no single “yes” score.

### Precise offer rules in this workshop

```mermaid
flowchart TD
    A[On-book account] --> B{Candidate probability ≥ 0.3121?}
    B -->|No| X[No offer]
    B -->|Yes| C{Origination PD ≤ 0.0741?}
    C -->|No| X
    C -->|Yes| D{Recommended amount > $0?}
    D -->|No| X
    D -->|Yes| E{Incremental ROE ≥ 15%?}
    E -->|No| X
    E -->|Yes| F[Eligible offer]
```

An account is `eligible` only when all four conditions are true:

1. **Candidate model probability ≥ 0.3121.** This is the model's probability that the
   account is a good line-increase candidate (`line_increase_good`), not the origination
   default PD.
2. **Origination PD ≤ 0.0741.** This is the risk-appetite ceiling, calibrated here to the
   lower-risk half of the on-book population. It prevents growth from going to the riskiest
   part of the book even when the candidate model is positive.
3. **Recommended amount > $0.** The amount must be economically meaningful after the
   utilization and exposure caps are applied.
4. **Incremental ROE ≥ 15%.** The additional exposure must clear the same ROE hurdle used
   in pricing. Incremental EAD is estimated as `recommended amount × current utilization`,
   then the shared pricing waterfall calculates incremental net income and ROE.

The recommended amount is deterministic:

```text
min(
    amount needed to reach 65% utilization,
    50% of current credit limit,
    30% of annual revenue − current credit limit
)
```

The `0.3121` probability threshold and `0.0741` PD ceiling are the committed values in
this trained workshop artifact, not universal bank standards. In a real portfolio they
would be calibrated, monitored, and approved as part of risk appetite. The important
business lesson is the intersection of **customer suitability, risk appetite, amount
discipline, and economics**. A next-best-action system that says “no” most of the time may
be doing its job.

In [ ]:
from line_increase.src import candidates as li_candidates

li_meta = json.loads(Path("line_increase/models/metadata.json").read_text())
li = li_candidates.score_population()
# An offer is what the platform calls an offer: `eligible`. This is exactly the
# filter line_increase.src.candidates.candidates() applies, so the notebook and
# the portal can never disagree.
offers = li[li["eligible"]]

print(f"on book, scored       : {len(li):,}")
print(f"offers made           : {len(offers):,}  ({len(offers)/len(li):.1%} of the book)")
print(f"total incremental line: ${offers['recommended_amount'].sum():,.0f}")
print(f"median incremental ROE: {offers['incremental_roe'].median():.1%}")
print(f"clears the hurdle     : {offers['clears_hurdle'].mean():.1%} of offers")
li.head()

In [ ]:
cols = ["recommended_amount", "incremental_roe", "pd"]
if len(offers) and cols:
    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3.6))
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, cols):
        ax.hist(offers[c], bins=35, color="#5b8ac6", edgecolor="white")
        ax.set_title(c)
    plt.tight_layout(); plt.show()

    print("The cohort test — is the cohort we grow SAFER than the book?")
    print(f"  offered cohort mean PD  : {offers['pd'].mean():.4f}")
    print(f"  whole book mean PD      : {li['pd'].mean():.4f}")
    print("  " + ("PASS — growth is going to the safer end of the book."
                  if offers['pd'].mean() < li['pd'].mean() else "FAIL — investigate."))
    print()
    print("  ...and are they people who will USE the line?")
    print(f"  offered cohort utilization: {offers['utilization_onbook'].mean():.4f}")
    print(f"  whole book utilization    : {li['utilization_onbook'].mean():.4f}")
    print("\n  Safer AND more utilised — that combination is the whole point:")
    print("  growth inside risk appetite, not growth by loosening it.")
else:
    print("No offers in this run.")

## One account, four decisions

Use the trace below to practice explaining the lifecycle to another person. The same
business can have a score band, a policy decision, a quoted rate, a watch status, and a
line-increase verdict. Each field answers a different question; none replaces the others.

In [ ]:
# Pick one business that appears in all four outputs and trace it end to end.
common_ids = set(dec["business_id"]) & set(priced["business_id"]) & set(ews["business_id"]) & set(li["business_id"])
sample_id = sorted(common_ids)[0]

d = dec.loc[dec["business_id"] == sample_id].iloc[0]
p = priced.loc[priced["business_id"] == sample_id].iloc[0]
e = ews.loc[ews["business_id"] == sample_id].iloc[0]
l = li.loc[li["business_id"] == sample_id].iloc[0]

trace = pd.Series({
    "business_id": sample_id,
    "score band": d["score_band"],
    "adjudication": d["decision"],
    "decision reasons": ", ".join(d["decision_reasons"]) if isinstance(d["decision_reasons"], list) else d["decision_reasons"],
    "quoted rate": p["quoted_rate"],
    "ROE at quoted rate": p["roe_at_quoted"],
    "watch tier": e["risk_tier"],
    "deterioration probability": e["prob"],
    "watch triggers": ", ".join(e["triggers"]),
    "line increase eligible": l["eligible"],
    "recommended amount": l["recommended_amount"],
    "incremental ROE": l["incremental_roe"],
})
trace.to_frame("value")

## Study prompts

1. If an applicant is in an AAA band, why might the loan still be mispriced?
2. Why is held-out top-decile capture more operationally useful than quoting AUC alone?
3. Which inputs are reused when deciding whether to offer a line increase?
4. Where could a policy change the business outcome without changing the score?
5. For the traced account, explain the difference between **risk**, **decision**, and
   **economics** in one sentence each.

### Core takeaway

Analytics pays off at the **decision endpoint**. A score ranks risk; policy decides access;
pricing decides whether access is profitable; monitoring decides where to spend attention;
line management decides whether growth is safe and worth funding.